<a href="https://colab.research.google.com/github/Carlos-A-Palomares/mis433/blob/main/ai_pet_breed_matchmaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Pet Breed Matchmaker

This Colab notebook works like a short personality test. It asks a user about their lifestyle and pet preferences, retrieves real dog and/or cat breed data from API Ninjas, then uses the OpenAI API to recommend exactly three pet breed matches with compatibility scores, images, explanations, drawbacks, and fun personality descriptions.

Assignment requirements covered:
- Uses a public data API: API Ninjas Dogs/Cats APIs
- Uses an AI API: OpenAI ChatGPT API
- Retrieves and displays useful data
- Performs light processing by matching user answers to candidate breeds
- Generates multiple AI outputs: ranked recommendations, compatibility scores, explanations, drawbacks, and creative pet profiles


In [1]:
# Install required packages. Run this once in Colab.
!pip -q install openai requests pandas

In [2]:
import json
import textwrap
import requests
import pandas as pd
from IPython.display import display, Markdown, Image
from google.colab import userdata
from openai import OpenAI

In [3]:
# API keys are stored in Colab Secrets.
# API Ninjas key name: ninja-pet
# OpenAI key name: OPENAI_API_KEY

NINJA_API_KEY = userdata.get("ninja-pet")
OPENAI_API_KEY = userdata.get("Chatgpt-MIS-433")

if not NINJA_API_KEY:
    raise ValueError("Missing API Ninjas key. Add it to Colab Secrets as 'ninja-pet'.")

if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI key. Add it to Colab Secrets as 'OPENAI_API_KEY'.")

client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = "chat-latest"

## Step 1: Pet Personality Test

Run the next cell and answer the prompts. The questions change slightly depending on whether the user prefers dogs, cats, or has no preference.

In [4]:
def ask_choice(question, valid_options):
    valid_options = [option.lower() for option in valid_options]
    while True:
        answer = input(question).strip().lower()
        if answer in valid_options:
            return answer
        print("Please choose one of:", ", ".join(valid_options))


def collect_user_profile():
    print("Welcome to the AI Pet Breed Matchmaker!\n")
    profile = {}

    profile["name"] = input("What is your name? ").strip().title()
    profile["pet_preference"] = ask_choice(
        "Do you prefer a dog, cat, or no preference? ",
        ["dog", "cat", "no preference"]
    )

    profile["activity_level"] = ask_choice(
        "How active are you? Choose low, medium, or high: ",
        ["low", "medium", "high"]
    )
    profile["home_type"] = ask_choice(
        "Do you live in an apartment, house, or dorm? ",
        ["apartment", "house", "dorm"]
    )
    profile["daily_time"] = ask_choice(
        "How much daily pet time can you give? Choose low, medium, or high: ",
        ["low", "medium", "high"]
    )
    profile["children"] = ask_choice(
        "Will the pet be around children? Choose yes or no: ",
        ["yes", "no"]
    )
    profile["other_pets"] = ask_choice(
        "Will the pet be around other pets? Choose yes or no: ",
        ["yes", "no"]
    )
    profile["grooming_tolerance"] = ask_choice(
        "How much grooming are you okay with? Choose low, medium, or high: ",
        ["low", "medium", "high"]
    )
    profile["shedding_tolerance"] = ask_choice(
        "How much shedding can you tolerate? Choose low, medium, or high: ",
        ["low", "medium", "high"]
    )
    profile["personality"] = ask_choice(
        "What pet personality sounds best? Choose calm, playful, affectionate, protective, or independent: ",
        ["calm", "playful", "affectionate", "protective", "independent"]
    )

    if profile["pet_preference"] in ["dog", "no preference"]:
        print("\nDog-specific follow-up questions:")
        profile["dog_size"] = ask_choice(
            "Preferred dog size? Choose small, medium, large, or no preference: ",
            ["small", "medium", "large", "no preference"]
        )
        profile["barking_tolerance"] = ask_choice(
            "How much barking is okay? Choose low, medium, or high: ",
            ["low", "medium", "high"]
        )
        profile["training_preference"] = ask_choice(
            "Do you want a very trainable dog? Choose yes or no: ",
            ["yes", "no"]
        )

    if profile["pet_preference"] in ["cat", "no preference"]:
        print("\nCat-specific follow-up questions:")
        profile["cat_playfulness"] = ask_choice(
            "Do you want a playful cat? Choose low, medium, or high: ",
            ["low", "medium", "high"]
        )
        profile["cat_affection"] = ask_choice(
            "Do you want a very affectionate cat? Choose yes or no: ",
            ["yes", "no"]
        )

    return profile


user_profile = collect_user_profile()
display(Markdown("### User Profile"))
display(pd.DataFrame([user_profile]).T.rename(columns={0: "answer"}))

Welcome to the AI Pet Breed Matchmaker!

What is your name? Carlos
Do you prefer a dog, cat, or no preference? dog
How active are you? Choose low, medium, or high: medium
Do you live in an apartment, house, or dorm? apartment
How much daily pet time can you give? Choose low, medium, or high: high
Will the pet be around children? Choose yes or no: yes
Will the pet be around other pets? Choose yes or no: yes
How much grooming are you okay with? Choose low, medium, or high: low
How much shedding can you tolerate? Choose low, medium, or high: low
What pet personality sounds best? Choose calm, playful, affectionate, protective, or independent: calm

Dog-specific follow-up questions:
Preferred dog size? Choose small, medium, large, or no preference: medium
How much barking is okay? Choose low, medium, or high: low
Do you want a very trainable dog? Choose yes or no: yes


### User Profile

,answer
name,Carlos
pet_preference,dog
activity_level,medium
home_type,apartment
daily_time,high
children,yes
other_pets,yes
grooming_tolerance,low
shedding_tolerance,low
personality,calm


## Step 2: Retrieve Breed Data From API Ninjas

The notebook uses dog and/or cat breed endpoints depending on the user's preference. It gathers a candidate pool, removes duplicates, and displays the raw breed data before asking OpenAI to rank the best matches.

In [6]:
DOG_BREEDS = [
    "golden retriever", "labrador retriever", "poodle", "cavalier king charles spaniel",
    "french bulldog", "shih tzu", "border collie", "greyhound", "chihuahua", "bernese mountain dog"
]

CAT_BREEDS = [
    "ragdoll", "maine coon", "persian", "siamese", "british shorthair",
    "abyssinian", "birman", "russian blue", "sphynx", "american shorthair"
]


def api_get(endpoint, params):
    url = f"https://api.api-ninjas.com/v1/{endpoint}"
    response = requests.get(url, headers={"X-Api-Key": NINJA_API_KEY}, params=params, timeout=20)
    response.raise_for_status()
    return response.json()


def fetch_breed_by_name(endpoint, breed_name):
    try:
        results = api_get(endpoint, {"name": breed_name})
        return results[0] if results else None
    except Exception as error:
        print(f"Could not fetch {breed_name}: {error}")
        return None


def annotate_pet_type(record, pet_type):
    if not record:
        return None
    record = dict(record)
    record["pet_type"] = pet_type
    return record


def build_candidate_pool(profile):
    candidates = []

    if profile["pet_preference"] in ["dog", "no preference"]:
        for breed in DOG_BREEDS:
            candidates.append(annotate_pet_type(fetch_breed_by_name("dogs", breed), "dog"))

    if profile["pet_preference"] in ["cat", "no preference"]:
        for breed in CAT_BREEDS:
            candidates.append(annotate_pet_type(fetch_breed_by_name("cats", breed), "cat"))

    candidates = [candidate for candidate in candidates if candidate]

    unique = {}
    for candidate in candidates:
        key = (candidate.get("pet_type"), candidate.get("name"))
        unique[key] = candidate

    return list(unique.values())


breed_candidates = build_candidate_pool(user_profile)

display(Markdown(f"### Retrieved {len(breed_candidates)} Breed Candidates"))
display(pd.DataFrame(breed_candidates))

### Retrieved 10 Breed Candidates

,image_link,good_with_children,good_with_other_dogs,shedding,grooming,drooling,coat_length,good_with_strangers,playfulness,protectiveness,...,max_height_male,max_height_female,max_weight_male,max_weight_female,min_height_male,min_height_female,min_weight_male,min_weight_female,name,pet_type
0,https://api-ninjas.com/images/dogs/golden_retr...,5,5,4,2,2,1,5,4,3,...,24.0,24.0,75.0,65.0,23.0,23.0,65.0,55.0,Golden Retriever,dog
1,https://api-ninjas.com/images/dogs/labrador_re...,5,5,4,2,2,1,5,5,3,...,24.5,24.5,80.0,70.0,22.5,22.5,65.0,55.0,Labrador Retriever,dog
2,https://api-ninjas.com/images/dogs/poodle_(min...,5,3,1,4,1,1,5,5,3,...,15.0,15.0,15.0,15.0,10.0,10.0,10.0,10.0,Poodle (Miniature),dog
3,https://api-ninjas.com/images/dogs/cavalier_ki...,5,5,2,2,2,1,4,3,3,...,13.0,13.0,18.0,18.0,12.0,12.0,13.0,13.0,Cavalier King Charles Spaniel,dog
4,https://api-ninjas.com/images/dogs/french_bull...,5,4,3,1,3,1,5,5,3,...,13.0,13.0,28.0,26.0,11.0,11.0,20.0,18.0,French Bulldog,dog
5,https://api-ninjas.com/images/dogs/shih_tzu.jpg,5,5,1,4,1,1,3,3,3,...,10.5,10.5,16.0,16.0,9.0,9.0,9.0,9.0,Shih Tzu,dog
6,https://api-ninjas.com/images/dogs/border_coll...,3,3,3,3,1,1,4,5,3,...,22.0,22.0,55.0,55.0,19.0,19.0,30.0,30.0,Border Collie,dog
7,https://api-ninjas.com/images/dogs/greyhound.jpg,3,4,2,1,1,1,3,3,3,...,30.0,30.0,70.0,65.0,28.0,28.0,65.0,60.0,Greyhound,dog
8,https://api-ninjas.com/images/dogs/chihuahua.jpg,1,3,2,1,1,2,2,4,4,...,8.0,8.0,6.0,6.0,5.0,5.0,4.0,4.0,Chihuahua,dog
9,https://api-ninjas.com/images/dogs/bernese_mou...,5,5,5,3,3,1,4,4,3,...,27.5,27.5,115.0,95.0,25.0,25.0,80.0,70.0,Bernese Mountain Dog,dog


## Step 3: Ask OpenAI To Recommend Exactly 3 Breeds

OpenAI receives the user's answers and only the breed data retrieved from the API. It must recommend exactly three options and return structured JSON so the notebook can display the results cleanly.

In [7]:
def recommend_with_openai(profile, candidates):
    prompt = f"""
You are helping with a college MIS project called AI Pet Breed Matchmaker.

Use the user's personality-test answers and the real breed data from API Ninjas.
Recommend exactly 3 breeds from the provided candidate list.

Rules:
- Only recommend breeds that appear in the candidate data.
- Rank the recommendations from best match to third best match.
- Give each breed a compatibility_score from 0 to 100.
- Include a clear reason that connects the user's answers to the breed traits.
- Include one possible drawback.
- Include a fun adoption-style personality_profile.
- Use the exact image_link from the breed data.
- Return only valid JSON with the key recommendations.

JSON format:
{{
  "recommendations": [
    {{
      "rank": 1,
      "breed_name": "Breed Name",
      "pet_type": "dog or cat",
      "compatibility_score": 95,
      "image_link": "https://...",
      "why_it_matches": "2-4 sentences",
      "possible_drawback": "1 sentence",
      "personality_profile": "2-3 fun sentences"
    }}
  ]
}}

User profile:
{json.dumps(profile, indent=2)}

Candidate breed data:
{json.dumps(candidates, indent=2)}
"""

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "developer", "content": "Return concise, helpful, student-project-friendly recommendations as valid JSON."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


ai_result = recommend_with_openai(user_profile, breed_candidates)
ai_result

{'recommendations': [{'rank': 1,
   'breed_name': 'French Bulldog',
   'pet_type': 'dog',
   'compatibility_score': 91,
   'image_link': 'https://api-ninjas.com/images/dogs/french_bulldog.jpg',
   'why_it_matches': 'This breed fits apartment living well and has a calm-to-moderate energy level that aligns with your medium activity lifestyle. It is great with children, generally does well with other dogs, has very low barking tendencies, and requires minimal grooming. Its trainability and friendly nature also match your preference for a manageable family companion.',
   'possible_drawback': 'French Bulldogs can be prone to breathing-related health issues, especially in hot weather.',
   'personality_profile': 'A charming apartment roommate who loves family time and attention. Friendly, playful, and happy to relax by your side after a short walk. Always ready to brighten the household with a goofy expression.'},
  {'rank': 2,
   'breed_name': 'Cavalier King Charles Spaniel',
   'pet_type'

## Step 4: Display Final Recommendations With Images

In [8]:
def display_recommendations(result):
    recommendations = result.get("recommendations", [])

    if len(recommendations) != 3:
        display(Markdown("**Note:** The model did not return exactly 3 recommendations. Check the JSON output above."))

    display(Markdown(f"# {user_profile['name']}'s Top Pet Breed Matches"))

    for rec in recommendations:
        title = f"## {rec.get('rank')}. {rec.get('breed_name')} ({rec.get('pet_type', '').title()}) - {rec.get('compatibility_score')}/100"
        display(Markdown(title))

        image_link = rec.get("image_link")
        if image_link:
            display(Image(url=image_link, width=320))

        display(Markdown("**Why it matches:** " + rec.get("why_it_matches", "")))
        display(Markdown("**Possible drawback:** " + rec.get("possible_drawback", "")))
        display(Markdown("**Personality profile:** " + rec.get("personality_profile", "")))
        display(Markdown("---"))


display_recommendations(ai_result)

# Carlos's Top Pet Breed Matches

## 1. French Bulldog (Dog) - 91/100

**Why it matches:** This breed fits apartment living well and has a calm-to-moderate energy level that aligns with your medium activity lifestyle. It is great with children, generally does well with other dogs, has very low barking tendencies, and requires minimal grooming. Its trainability and friendly nature also match your preference for a manageable family companion.

**Possible drawback:** French Bulldogs can be prone to breathing-related health issues, especially in hot weather.

**Personality profile:** A charming apartment roommate who loves family time and attention. Friendly, playful, and happy to relax by your side after a short walk. Always ready to brighten the household with a goofy expression.

...

## 3. Golden Retriever (Dog) - 82/100

**Why it matches:** Golden Retrievers are exceptionally trainable, wonderful with children, and highly compatible with other dogs. Their calm and friendly temperament aligns well with your personality preferences, and their low barking score is a strong plus. The breed also benefits from the high amount of daily time you can provide.

**Possible drawback:** This breed sheds heavily and is larger than your preferred medium size.

**Personality profile:** A loyal family favorite who loves making friends and learning new things. Patient, dependable, and eager to please, it thrives on positive interaction. Expect a happy companion who treats everyone like a future best friend.

---